# Bozyazı deniz seviyesi analizi — Colab

Gelgit ve gelgit dışı bileşenlerin ayrıştırılması, gelgit düzeyleri ve
PDF/CDF figürleri.

**Önce çalışma zamanını yüksek RAM'e alın.** `05_harmonik_analiz.py`
15 dakikalık çözünürlükte 16 yıllık kayıtla çalışırken UTide'ın en küçük
kareler tasarım matrisi ~9 GB istiyor. Standart Colab (12 GB) sınırda
kalır; yüksek-RAM çalışma zamanı rahat çalıştırır.

`Çalışma zamanı → Çalışma zamanı türünü değiştir → Yüksek RAM`

## 1. Kurulum

In [ ]:
!git clone -q https://github.com/adzetto/marine_analysis.git
%cd marine_analysis/su-seviyesi
!pip install -q -r requirements.txt

LaTeX dizgisi isteğe bağlı. Kurulmazsa figürler matplotlib'in kendi
matematik dizgisiyle üretilir; betikler bunu kendisi algılıyor. Yayın
kalitesi isteniyorsa aşağıyı çalıştırın (birkaç dakika).

In [ ]:
!apt-get -qq update && apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super > /dev/null
print('latex kuruldu')

## 2. Kaynakları göster

Harmonik çözümün sığıp sığmayacağını baştan görmek için.

In [ ]:
import psutil, os
gb = psutil.virtual_memory().total / 1e9
print(f'toplam RAM : {gb:.1f} GB')
print(f'CPU        : {os.cpu_count()} cekirdek')
if gb < 20:
    print('\nUYARI: 05_harmonik_analiz.py tum kayitta ~9 GB istiyor.')
    print('Yuksek RAM calisma zamanina gecmeniz onerilir.')

## 3. Veri

**Veri depoyla birlikte geliyor** (`data/bozyazi_ham.dat.gz` ve
`data/bozyazi_temiz.dat.gz`), indirme adımına gerek yok.

Sebebi teknik: `tudes.harita.gov.tr` Türkiye dışındaki DNS
çözümleyicilerinden ad çözümlemesi yapmıyor — 1.1.1.1 ve 9.9.9.9 çözemiyor,
Colab da çözemiyor. Bu yüzden `01_tudes_indir.py` bulutta
`Temporary failure in name resolution` ile duruyor. İndirme yalnızca
Türkiye'den erişilen bir makinede çalışır; veri orada indirilip depoya
konuyor.

Aşağıdaki hücre elinizdekini doğrular.

In [ ]:
from ortak import oku, veri_yolu
for ad in ('bozyazi_ham.dat', 'bozyazi_temiz.dat'):
    print(f'{ad:<22} -> {veri_yolu(ad).name}')
s = oku()
print(f'\nayiklanmis seri : {len(s):,} kayit')
print(f'aralik          : {s.index.min()} -> {s.index.max()}')
print(f'seviye          : {s.min():.3f} - {s.max():.3f} m, '
      f'ortalama {s.mean():.4f} m')

Veriyi sıfırdan üretmek isterseniz (yalnız Türkiye'den erişilen bir
makinede çalışır) aşağıdaki satırları açın:

In [ ]:
# !python -u 01_tudes_indir.py
# !python -u 02_veri_birlestir.py

## 4. Ayıklama

Sıçrama, sürekli blok ve takılmış sensör ölçümlerini atar, 1 günden kısa
boşlukları doldurur, MSL'i hesaplar. Ardından ayıklamanın doğru şeyi
sildiği sınanır: silinenler bozuk yıllarda toplanmalı, hocanın gösterdiği
Temmuz/Eylül 2025 anomalileri tam olarak kalkmalı.

Depodaki `bozyazi_temiz.dat.gz` zaten bu adımın çıktısı; yeniden çalıştırmak
üzerine yazar.

In [ ]:
!python -u 03_veri_ayikla.py
!python -u 04_ayiklama_dogrula.py

## 5. Harmonik analiz

Ağır adım bu. Dört dönem için UTide çözümü, yayımlanmış Bozyazı
değerleriyle karşılaştırma ve deniz seviyesi trendi.

In [ ]:
!python -u 05_harmonik_analiz.py

## 6. Gelgit düzeyleri ve gelgit dışı bileşen

In [ ]:
!python -u 06_gelgit_seviyeleri.py
!python -u 07_non_tidal.py

## 7. Sonuçları göster

In [ ]:
import pandas as pd, glob
from IPython.display import display, Image

for f in sorted(glob.glob('tables/*.csv')):
    print('=' * 70); print(f); print('=' * 70)
    display(pd.read_csv(f))

for f in sorted(glob.glob('figures/*.png')):
    print(f)
    display(Image(f))

## 8. Çıktıları indir

In [ ]:
!zip -qr bozyazi_sonuclar.zip tables figures data
from google.colab import files
files.download('bozyazi_sonuclar.zip')